In [2]:
# Day 10 - Flight Operations Data Analysis
# full workflow: load -> inspect -> filter -> sort -> groupby -> transform -> findings

import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

df = pd.read_csv('Day10_Flight_Operations_Dataset.csv')
df.head()

,Flight_ID,Flight_Date,Airline,Origin,Destination,Aircraft,Travel_Class,Passengers,Seat_Capacity,Average_Ticket_Price,Delay_Minutes,Flight_Status,Weather,Booking_Channel,Avg_Baggage_Kg,Meal_Preference,Passenger_Satisfaction
0,FL0001,2026-03-23,SpiceJet,Mumbai,Bengaluru,Airbus A319,Economy,210,180,6894,50.0,Delayed,Storm,Online Travel Portal,23,No Meal,3
1,FL0002,2026-06-07,Akasa Air,Delhi,Mumbai,Boeing 737,Economy,70,210,4468,0.0,On Time,Fog,Travel Agency,16,Vegan,5
2,FL0003,2026-05-12,Air India,Kochi,Delhi,Boeing 737,Economy,67,220,4713,5.0,On Time,Rain,Travel Agency,19,No Meal,2
3,FL0004,2026-05-30,Akasa Air,Srinagar,Delhi,Airbus A319,Economy,153,220,3791,35.0,Delayed,Storm,Travel Agency,18,No Meal,2
4,FL0005,2026-02-27,Air India,Pune,Delhi,Airbus A319,Premium Economy,178,180,6505,20.0,Delayed,Cloudy,Airline Website,25,No Meal,5


In [3]:
# basic structure of the dataset
print(df.shape)
df.info()

(180, 17)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Flight_ID               180 non-null    object 
 1   Flight_Date             180 non-null    object 
 2   Airline                 180 non-null    object 
 3   Origin                  180 non-null    object 
 4   Destination             180 non-null    object 
 5   Aircraft                180 non-null    object 
 6   Travel_Class            180 non-null    object 
 7   Passengers              180 non-null    int64  
 8   Seat_Capacity           180 non-null    int64  
 9   Average_Ticket_Price    180 non-null    int64  
 10  Delay_Minutes           173 non-null    float64
 11  Flight_Status           180 non-null    object 
 12  Weather                 180 non-null    object 
 13  Booking_Channel         180 non-null    object 
 14  Avg_Baggage_Kg          180 non-

In [4]:
# summary statistics for the numeric columns
df.describe()

,Passengers,Seat_Capacity,Average_Ticket_Price,Delay_Minutes,Avg_Baggage_Kg,Passenger_Satisfaction
count,180.000000,180.000000,180.000000,173.000000,180.000000,180.000000
mean,122.772222,197.238889,6111.677778,19.335260,15.150000,3.494444
std,45.635651,15.408176,3615.220040,29.029712,5.705506,1.169908
min,45.000000,180.000000,1949.000000,0.000000,5.000000,2.000000
25%,78.750000,186.000000,3815.500000,0.000000,10.000000,2.000000
50%,125.000000,189.000000,5380.000000,5.000000,15.500000,3.000000
75%,162.250000,210.000000,6790.000000,20.000000,20.000000,5.000000
max,210.000000,220.000000,27832.000000,110.000000,25.000000,5.000000


In [5]:
# checking for missing values
df.isna().sum()

,0
Flight_ID,0
Flight_Date,0
Airline,0
Origin,0
Destination,0
Aircraft,0
Travel_Class,0
Passengers,0
Seat_Capacity,0
Average_Ticket_Price,0


In [6]:
# the only nulls are in Delay_Minutes - makes sense, cancelled flights don't have a delay logged
df[df['Delay_Minutes'].isna()]['Flight_Status'].value_counts()

,count
Flight_Status,
Cancelled,7


In [7]:
# filling those with 0 since a cancelled flight has no delay time to report
df['Delay_Minutes'] = df['Delay_Minutes'].fillna(0)
df.isna().sum().sum()

np.int64(0)

In [8]:
# converting date column to datetime so we can do date-based ops later
df['Flight_Date'] = pd.to_datetime(df['Flight_Date'])
df['Month'] = df['Flight_Date'].dt.month_name()
df['Weekday'] = df['Flight_Date'].dt.day_name()

In [9]:
# --- selecting & filtering ---

# all delayed flights
delayed = df[df['Flight_Status'] == 'Delayed']
print(len(delayed))

# flights delayed by more than 30 mins in bad weather
bad_weather_delays = df[(df['Delay_Minutes'] > 30) & (df['Weather'].isin(['Storm', 'Fog', 'Rain']))]
bad_weather_delays[['Flight_ID', 'Airline', 'Weather', 'Delay_Minutes']].head()

59


,Flight_ID,Airline,Weather,Delay_Minutes
0,FL0001,SpiceJet,Storm,50.0
3,FL0004,Akasa Air,Storm,35.0
29,FL0030,SpiceJet,Storm,75.0
43,FL0044,Air India Express,Storm,35.0
50,FL0051,Air India,Rain,110.0


In [10]:
# business class flights with low satisfaction (something worth flagging)
low_satisfaction_business = df[(df['Travel_Class'] == 'Business') & (df['Passenger_Satisfaction'] <= 2)]
low_satisfaction_business[['Flight_ID', 'Airline', 'Passenger_Satisfaction']]

,Flight_ID,Airline,Passenger_Satisfaction
51,FL0052,SpiceJet,2
52,FL0053,Akasa Air,2
92,FL0093,Air India Express,2
108,FL0109,SpiceJet,2


In [11]:
# --- sorting ---

# top 10 most delayed flights
df.sort_values('Delay_Minutes', ascending=False)[['Flight_ID', 'Airline', 'Delay_Minutes', 'Weather']].head(10)

,Flight_ID,Airline,Delay_Minutes,Weather
50,FL0051,Air India,110.0,Rain
26,FL0027,SpiceJet,110.0,Clear
82,FL0083,Air India Express,110.0,Cloudy
57,FL0058,IndiGo,110.0,Cloudy
177,FL0178,IndiGo,110.0,Fog
128,FL0129,SpiceJet,110.0,Cloudy
169,FL0170,IndiGo,110.0,Rain
161,FL0162,Air India,110.0,Rain
146,FL0147,IndiGo,110.0,Rain
21,FL0022,Vistara,75.0,Clear


In [12]:
# flights sorted by ticket price, most expensive first
df.sort_values('Average_Ticket_Price', ascending=False)[['Flight_ID', 'Airline', 'Travel_Class', 'Average_Ticket_Price']].head(10)

,Flight_ID,Airline,Travel_Class,Average_Ticket_Price
138,FL0139,Air India Express,Business,27832
52,FL0053,Akasa Air,Business,21499
32,FL0033,Akasa Air,Business,17625
176,FL0177,Vistara,Business,17306
108,FL0109,SpiceJet,Business,16390
132,FL0133,IndiGo,Business,15324
155,FL0156,Vistara,Business,15040
7,FL0008,Air India,Business,14173
51,FL0052,SpiceJet,Business,13824
92,FL0093,Air India Express,Business,12970


In [13]:
# --- grouping & aggregation ---

# average delay per airline
df.groupby('Airline')['Delay_Minutes'].mean().sort_values(ascending=False)

,Delay_Minutes
Airline,
IndiGo,25.031250
SpiceJet,21.875000
Air India,20.375000
Vistara,17.621622
Akasa Air,13.935484
Air India Express,11.291667


In [14]:
# average passenger satisfaction per travel class
df.groupby('Travel_Class')['Passenger_Satisfaction'].mean().sort_values(ascending=False)

,Passenger_Satisfaction
Travel_Class,
Premium Economy,3.685714
Economy,3.462687
Business,3.272727


In [15]:
# flight count and average delay per weather condition
df.groupby('Weather').agg(
    Flights=('Flight_ID', 'count'),
    Avg_Delay=('Delay_Minutes', 'mean'),
    Avg_Satisfaction=('Passenger_Satisfaction', 'mean')
).sort_values('Avg_Delay', ascending=False)

,Flights,Avg_Delay,Avg_Satisfaction
Weather,,,
Rain,43,21.186047,3.372093
Storm,30,19.433333,3.166667
Cloudy,38,18.447368,3.842105
Clear,34,18.029412,3.352941
Fog,35,15.342857,3.685714


In [16]:
# busiest routes (origin -> destination)
df.groupby(['Origin', 'Destination'])['Flight_ID'].count().sort_values(ascending=False).head(10)

Origin     Destination
Delhi      Chennai        17
Mumbai     Bengaluru      16
Srinagar   Delhi          15
Kochi      Delhi          14
Delhi      Srinagar       13
           Mumbai         13
Bengaluru  Hyderabad      13
Mumbai     Kolkata        13
Ahmedabad  Bengaluru      12
Lucknow    Delhi          11
Name: Flight_ID, dtype: int64

In [17]:
# average revenue per flight (price * passengers) grouped by airline
df['Revenue'] = df['Average_Ticket_Price'] * df['Passengers']
df.groupby('Airline')['Revenue'].mean().sort_values(ascending=False)

,Revenue
Airline,
Air India Express,997673.500000
IndiGo,803410.343750
Vistara,775391.000000
SpiceJet,756122.968750
Air India,676350.750000
Akasa Air,608776.096774


In [18]:
# --- transformations ---

# load factor = how full the flight was (passengers / seat capacity)
df['Load_Factor_%'] = (df['Passengers'] / df['Seat_Capacity'] * 100).round(1)

# simple delay category using apply
def delay_category(mins):
    if mins == 0:
        return 'No Delay'
    elif mins <= 15:
        return 'Minor'
    elif mins <= 45:
        return 'Moderate'
    else:
        return 'Severe'

df['Delay_Category'] = df['Delay_Minutes'].apply(delay_category)
df[['Flight_ID', 'Passengers', 'Seat_Capacity', 'Load_Factor_%', 'Delay_Minutes', 'Delay_Category']].head()

,Flight_ID,Passengers,Seat_Capacity,Load_Factor_%,Delay_Minutes,Delay_Category
0,FL0001,210,180,116.7,50.0,Severe
1,FL0002,70,210,33.3,0.0,No Delay
2,FL0003,67,220,30.5,5.0,Minor
3,FL0004,153,220,69.5,35.0,Moderate
4,FL0005,178,180,98.9,20.0,Moderate


In [19]:
df['Delay_Category'].value_counts()

,count
Delay_Category,
No Delay,77
Minor,44
Moderate,32
Severe,27


In [20]:
# average load factor by airline - who's flying fuller planes
df.groupby('Airline')['Load_Factor_%'].mean().sort_values(ascending=False)

,Load_Factor_%
Airline,
IndiGo,66.778125
Air India Express,64.791667
Air India,64.662500
SpiceJet,63.600000
Vistara,61.867568
Akasa Air,55.206452


In [21]:
# does booking channel affect satisfaction?
df.groupby('Booking_Channel')['Passenger_Satisfaction'].mean().sort_values(ascending=False)

,Passenger_Satisfaction
Booking_Channel,
Online Travel Portal,3.740000
Mobile App,3.444444
Travel Agency,3.386364
Airline Website,3.365854


In [22]:
# monthly trend - flights and average delay per month
df.groupby('Month').agg(
    Flights=('Flight_ID', 'count'),
    Avg_Delay=('Delay_Minutes', 'mean')
).reindex(['January', 'February', 'March', 'April', 'May', 'June'])

,Flights,Avg_Delay
Month,,
January,29,22.103448
February,28,22.750000
March,33,9.393939
April,35,24.685714
May,30,15.800000
June,25,16.760000


In [23]:
# final look at the enriched dataset
df.head()

,Flight_ID,Flight_Date,Airline,Origin,Destination,Aircraft,Travel_Class,Passengers,Seat_Capacity,Average_Ticket_Price,Delay_Minutes,Flight_Status,Weather,Booking_Channel,Avg_Baggage_Kg,Meal_Preference,Passenger_Satisfaction,Month,Weekday,Revenue,Load_Factor_%,Delay_Category
0,FL0001,2026-03-23,SpiceJet,Mumbai,Bengaluru,Airbus A319,Economy,210,180,6894,50.0,Delayed,Storm,Online Travel Portal,23,No Meal,3,March,Monday,1447740,116.7,Severe
1,FL0002,2026-06-07,Akasa Air,Delhi,Mumbai,Boeing 737,Economy,70,210,4468,0.0,On Time,Fog,Travel Agency,16,Vegan,5,June,Sunday,312760,33.3,No Delay
2,FL0003,2026-05-12,Air India,Kochi,Delhi,Boeing 737,Economy,67,220,4713,5.0,On Time,Rain,Travel Agency,19,No Meal,2,May,Tuesday,315771,30.5,Minor
3,FL0004,2026-05-30,Akasa Air,Srinagar,Delhi,Airbus A319,Economy,153,220,3791,35.0,Delayed,Storm,Travel Agency,18,No Meal,2,May,Saturday,580023,69.5,Moderate
4,FL0005,2026-02-27,Air India,Pune,Delhi,Airbus A319,Premium Economy,178,180,6505,20.0,Delayed,Cloudy,Airline Website,25,No Meal,5,February,Friday,1157890,98.9,Moderate


In [24]:
# saving the processed version
df.to_csv('Flight_Operations_Processed.csv', index=False)
print('saved! rows:', len(df))

saved! rows: 180


In [25]:

# --- Observations ---
# 1. IndiGo has the highest average delay (~25 min) of all airlines, followed by SpiceJet and Air India,
#    while Air India Express and Akasa Air run closest to schedule.
# 2. Rain causes the highest average delay, slightly ahead of Storm - Fog actually has the lowest average
#    delay of all weather types, which is a bit counter-intuitive.
# 3. Premium Economy passengers report the highest average satisfaction, not Business - Business class
#    is actually the lowest of the three, so a higher fare doesn't guarantee a happier passenger here.
# 4. Cancelled flights make up under 4% of the dataset, and all of them have no delay value logged
#    since a cancelled flight never departs.
# 5. Delhi-Chennai and Mumbai-Bengaluru are the busiest routes by flight count.
# 6. IndiGo also has the highest average load factor (fullest planes), while Akasa Air runs the emptiest
#    planes on average - about 11 percentage points lower.
# 7. Passengers booking through the Online Travel Portal report the highest average satisfaction,
#    while those booking directly on the Airline Website report the lowest - the opposite of what
#    you'd expect if going direct was assumed to be better.
# 8. Severe delays (45+ min) are fairly evenly spread across weather types rather than concentrated in
#    Storm/Fog alone, suggesting weather isn't the only factor driving the worst delays.
